# Historical research notebook
Run from the repository root so data/raw paths resolve. Review docs/evaluation-audit.md first.
Original cell order preserved; historical outputs extracted to docs/historical-results.txt.
Cleared execution counters do not establish clean-kernel reproducibility.


In [ ]:
from pathlib import Path
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs" / "evaluation-audit.md").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from within market-anomaly-research.")
DATA_DIR = REPO_ROOT / "data" / "raw"


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from tqdm import tqdm
import yfinance as yf
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

In [ ]:
# FinBERT 감성분석 파이프라인 준비
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
finbert = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

In [ ]:
# 주가 및 뉴스 대상
tickers = {
    "TRBO": { "anomaly_start": "30-Mar-20", "anomaly_end": "9-Apr-20"  },
    "APPB": { "anomaly_start": "25-Mar-20", "anomaly_end": "13-Apr-20" },
    "AEMD": { "anomaly_start": "22-Jan-20", "anomaly_end": "7-Feb-20"  },
    "NBDR": { "anomaly_start": "11-Mar-20", "anomaly_end": "3-Apr-20"  },
    "GME" : { "anomaly_start": "11-Jan-21", "anomaly_end": "29-Jan-21" }
}

# 주가와 뉴스 csv 경로 지정
stock_csv_files = {
    'AEMD': str(DATA_DIR / "AEMD_augmented.csv"),
    'APPB': str(DATA_DIR / "APPB_augmented.csv"),
    'GME':  str(DATA_DIR / "GME_augmented.csv"),
    'NBDR': str(DATA_DIR / "NBDR_augmented.csv"),
    'TRBO': str(DATA_DIR / "TRBO_augmented.csv"),
}

In [ ]:
# 일별 감성평균
def daily_sentiment(news_df):
    df = news_df.groupby('Date')['sentiment_score'].mean().reset_index()
    df.rename(columns={'sentiment_score': 'sentiment_mean'}, inplace=True)
    return df

In [ ]:
# 감성 이상치 탐지 (z-score 방식)
def detect_sentiment_anomaly(df_sent, z_thresh=0.5):
    mean = df_sent['sentiment_mean'].mean()
    std = df_sent['sentiment_mean'].std()
    df_sent['zscore'] = (df_sent['sentiment_mean'] - mean) / std
    df_sent['sentiment_anomaly'] = (df_sent['zscore'].abs() > z_thresh).astype(int)
    return df_sent

In [ ]:
# yfinance에서 주가 시계열 수집 (기간 자동계산)
import datetime
def get_date_minus_months(date_str, months):
    dt = datetime.datetime.strptime(date_str, "%Y-%m-%d")
    year, month = dt.year, dt.month - months
    while month <= 0:
        year -= 1
        month += 12
    day = dt.day
    try: new_dt = datetime.datetime(year, month, day)
    except ValueError: new_dt = datetime.datetime(year, month, 1) + pd.offsets.MonthEnd(0)
    return new_dt.strftime("%Y-%m-%d")

def get_date_plus_months(date_str, months):
    dt = datetime.datetime.strptime(date_str, "%Y-%m-%d")
    year, month = dt.year, dt.month + months
    while month > 12:
        year += 1
        month -= 12
    day = dt.day
    try: new_dt = datetime.datetime(year, month, day)
    except ValueError: new_dt = datetime.datetime(year, month, 1) + pd.offsets.MonthEnd(0)
    return new_dt.strftime("%Y-%m-%d")

In [ ]:
# 논문 스타일 시각화 함수
def plot_anomaly_detection(
    all_stock_data,     # {stock: DataFrame('Date', 'Volume')}
    all_anomalies,      # {stock: DataFrame('start', 'end')} (unixtime)
    detected_anomalies, # {stock: [ (start_date, end_date), ... ] }
    result_df           # DataFrame: Stock, Precision, Recall, F1-score
):
    fig, axs = plt.subplots(5, 1, figsize=(9, 12), sharex=True)
    for idx, stock in enumerate(all_stock_data.keys()):
        ax = axs[idx]
        df = all_stock_data[stock]
        ax.plot(df['Date'], df['Volume'], label='Volume', color='royalblue', lw=1)
        for _, row in all_anomalies[stock].iterrows():
            ax.axvspan(pd.to_datetime(row['start'], unit='s'), pd.to_datetime(row['end'], unit='s'), color='green', alpha=0.2)
        for start, end in detected_anomalies.get(stock, []):
            ax.axvspan(start, end, color='red', alpha=0.18)
        company = {
            "TRBO": "Turbo Global Partners, Inc.",
            "APPB": "Applied Biosciences Corp",
            "AEMD": "Aethlon Medical, Inc.",
            "NBDR": "No Borders, Inc.",
            "GME": "GameStop"
        }[stock]
        ax.set_title(f'Traded volumes for {company} (“{stock}”)', fontsize=11)
        ax.set_ylabel('Volume')
        if idx == 0:
            green_patch = mpatches.Patch(color='green', alpha=0.2, label='Real anomalies')
            red_patch = mpatches.Patch(color='red', alpha=0.18, label='Detected anomalies')
            ax.legend(handles=[red_patch, green_patch], loc='upper left', fontsize=8)
    axs[-1].set_xlabel('Time')
    plt.tight_layout(rect=[0, 0.1, 1, 1])
    cell_text = []
    for _, row in result_df.iterrows():
        cell_text.append([row['Stock'], f"{row['Precision']:.2f}", f"{row['Recall']:.2f}", f"{row['F1-score']:.2f}"])
    table = plt.table(
        cellText=cell_text,
        colLabels=["Stock", "Precision", "Recall", "F1-score"],
        cellLoc='center',
        loc='bottom',
        bbox=[0.15, -0.27, 0.7, 0.15]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    plt.subplots_adjust(bottom=0.21)
    plt.figtext(0.5, 0.01, "Fig. 2. Best Results for Each Data Set", ha='center', va='center', fontsize=13)
    plt.show()

In [ ]:
def run_finbert(news_df):
    tqdm.pandas()
    # ① 'text' 컬럼을 먼저 만듭니다! (Headline + Summary)
    news_df['text'] = news_df['Headline'].fillna('') + '. ' + news_df['Summary'].fillna('')
    # ② 감성 분석
    news_df['sentiment'] = news_df['text'].progress_apply(lambda x: finbert(x)[0]['label'])
    sentiment_map = {'positive': 1, 'neutral': 0, 'negative': -1}
    news_df['sentiment_score'] = news_df['sentiment'].map(sentiment_map)
    return news_df

In [ ]:
# ========== 메인 파이프라인 ==========

all_stock_data = {}
all_anomalies = {}
detected_anomalies = {}
result_metrics = []
df_sent_dict = {}

for stock, info in tickers.items():
    # 1. 실제 이상치 구간 변환
    anom_start_str = pd.to_datetime(info["anomaly_start"], format="%d-%b-%y").strftime("%Y-%m-%d")
    anom_end_str   = pd.to_datetime(info["anomaly_end"],   format="%d-%b-%y").strftime("%Y-%m-%d")
    hist_start = get_date_minus_months(anom_start_str, 24)
    hist_end   = get_date_plus_months(anom_end_str, 12)

    # 2. 주가 시계열 수집
    stock_data = yf.Ticker(stock)
    df = stock_data.history(start=hist_start, end=hist_end).reset_index()
    df['Date'] = pd.to_datetime(df['Date'])
    all_stock_data[stock] = df[['Date', 'Volume']]

    # 3. 실제 이상치 구간(unix timestamp)
    anom_start_ts = int(pd.Timestamp(anom_start_str).timestamp())
    anom_end_ts   = int(pd.Timestamp(anom_end_str).timestamp())
    all_anomalies[stock] = pd.DataFrame([[anom_start_ts, anom_end_ts]], columns=['start','end'])

    # 4. 뉴스기사 감성분석
    news = pd.read_csv(stock_csv_files[stock], encoding='latin1', on_bad_lines='skip')
    news['Date'] = pd.to_datetime(news['Date']).dt.date
    news = run_finbert(news)
    df_sent = daily_sentiment(news)
    df_sent = detect_sentiment_anomaly(df_sent)
    df_sent['Date'] = pd.to_datetime(df_sent['Date'])

    df_sent_dict[stock] = df_sent

    # 5. 감성 이상치 탐지 구간(빨간색 음영)
    detected = []
    grouped = df_sent[df_sent['sentiment_anomaly'] == 1].groupby((df_sent['sentiment_anomaly'] != df_sent['sentiment_anomaly'].shift()).cumsum())
    for _, group in grouped:
        if group['sentiment_anomaly'].iloc[0] == 1:
            start = group['Date'].min()
            end = group['Date'].max()
            detected.append((start, end))
    detected_anomalies[stock] = detected

    # 6. 정답라벨 생성
    s = pd.to_datetime(info['anomaly_start'], format="%d-%b-%y")
    e = pd.to_datetime(info['anomaly_end'], format="%d-%b-%y")
    df_sent['label'] = ((df_sent['Date'] >= s) & (df_sent['Date'] <= e)).astype(int)
    precision = precision_score(df_sent['label'], df_sent['sentiment_anomaly'], zero_division=0)
    recall    = recall_score(df_sent['label'], df_sent['sentiment_anomaly'], zero_division=0)
    f1        = f1_score(df_sent['label'], df_sent['sentiment_anomaly'], zero_division=0)
    result_metrics.append([stock, round(precision,3), round(recall,3), round(f1,3)])


In [ ]:
# 7. 결과표 DataFrame
result_df = pd.DataFrame(result_metrics, columns=['Stock','Precision','Recall','F1-score'])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates

def plot_volumes_with_anomalies(
    all_stock_data,     # {stock: DataFrame('Date', 'Volume')}
    all_anomalies,      # {stock: DataFrame('start', 'end')} (unixtime)
    detected_anomalies  # {stock: [ (start_date, end_date), ... ] }
):
    stocks = list(all_stock_data.keys())
    n_stocks = len(stocks)
    fig, axs = plt.subplots(n_stocks, 1, figsize=(10, 12), sharex=True)

    for idx, stock in enumerate(stocks):
        ax = axs[idx]
        df = all_stock_data[stock]

        # 1. 거래량 시계열
        ax.plot(df['Date'], df['Volume'], label='Volume', color='royalblue', lw=1)

        # 2. 실제 이상치 구간 (녹색 음영)
        for _, row in all_anomalies[stock].iterrows():
            ax.axvspan(
                pd.to_datetime(row['start'], unit='s'),
                pd.to_datetime(row['end'], unit='s'),
                color='green', alpha=0.2
            )

        # 3. 감성분석 이상치 구간 (빨간색 음영)
        for start, end in detected_anomalies.get(stock, []):
            ax.axvspan(
                start, end,
                color='red', alpha=0.18
            )

        # 4. 제목 및 레이블
        company = {
            "TRBO": "Turbo Global Partners, Inc.",
            "APPB": "Applied Biosciences Corp",
            "AEMD": "Aethlon Medical, Inc.",
            "NBDR": "No Borders, Inc.",
            "GME": "GameStop"
        }.get(stock, stock)
        ax.set_title(f'Traded volumes for {company} (“{stock}”)', fontsize=11)
        ax.set_ylabel('Volume')

        # 5. 첫번째 그래프에만 범례 추가
        if idx == 0:
            green_patch = mpatches.Patch(color='green', alpha=0.2, label='Real anomalies')
            red_patch = mpatches.Patch(color='red', alpha=0.18, label='Detected anomalies')
            ax.legend(handles=[red_patch, green_patch], loc='upper left', fontsize=8)

        # x축 날짜 포맷 지정 (예: 2020-01)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.grid(True)

    axs[-1].set_xlabel('Date')
    plt.tight_layout(rect=[0, 0, 1, 1])
    plt.show()


In [ ]:
plot_volumes_with_anomalies(all_stock_data, all_anomalies, detected_anomalies)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates

def plot_anomaly_detection(
    all_stock_data,     # {stock: DataFrame('Date', 'Volume')}
    all_anomalies,      # {stock: DataFrame('start', 'end')} (unixtime)
    detected_anomalies, # {stock: [ (start_date, end_date), ... ] }
    result_df           # DataFrame: Stock, Precision, Recall, F1-score
):
    stocks = list(all_stock_data.keys())
    n_stocks = len(stocks)
    fig, axs = plt.subplots(n_stocks, 1, figsize=(10, 12), sharex=True)

    for idx, stock in enumerate(stocks):
        ax = axs[idx]
        df = all_stock_data[stock]

        # 1. 거래량 시계열
        ax.plot(df['Date'], df['Volume'], label='Volume', color='royalblue', lw=1)

        # 2. 실제 이상치 구간 (녹색 음영)
        for _, row in all_anomalies[stock].iterrows():
            ax.axvspan(
                pd.to_datetime(row['start'], unit='s'),
                pd.to_datetime(row['end'], unit='s'),
                color='green', alpha=0.2
            )

        # 3. 감성분석 이상치 구간 (빨간색 음영)
        for start, end in detected_anomalies.get(stock, []):
            ax.axvspan(
                start, end,
                color='red', alpha=0.18
            )

        # 4. 제목 및 레이블
        company = {
            "TRBO": "Turbo Global Partners, Inc.",
            "APPB": "Applied Biosciences Corp",
            "AEMD": "Aethlon Medical, Inc.",
            "NBDR": "No Borders, Inc.",
            "GME": "GameStop"
        }.get(stock, stock)
        ax.set_title(f'Traded volumes for {company} (“{stock}”)', fontsize=11)
        ax.set_ylabel('Volume')

        # 5. 첫번째 그래프에만 범례 추가
        if idx == 0:
            green_patch = mpatches.Patch(color='green', alpha=0.2, label='Real anomalies')
            red_patch = mpatches.Patch(color='red', alpha=0.18, label='Detected anomalies')
            ax.legend(handles=[red_patch, green_patch], loc='upper left', fontsize=8)

        # x축 날짜 포맷 지정 (예: 2020-01)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.grid(True)

    axs[-1].set_xlabel('Date')
    plt.tight_layout(rect=[0, 0.13, 1, 1])

    # 6. F1-score 표 추가 (아래쪽)
    cell_text = []
    for _, row in result_df.iterrows():
        cell_text.append([
            row['Stock'],
            f"{row['Precision']:.2f}",
            f"{row['Recall']:.2f}",
            f"{row['F1-score']:.2f}"
        ])
    table = plt.table(
        cellText=cell_text,
        colLabels=["Stock", "Precision", "Recall", "F1-score"],
        cellLoc='center',
        loc='bottom',
        bbox=[0.15, -0.24, 0.7, 0.13]  # (left, bottom, width, height)
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    plt.subplots_adjust(bottom=0.20)
    plt.figtext(0.5, 0.02, "Fig. 2. Best Results for Each Data Set", ha='center', va='center', fontsize=13)
    plt.show()


In [ ]:
print(result_df)

In [ ]:
print(df_sent['label'].value_counts())
print(df_sent['sentiment_anomaly'].value_counts())
print(df_sent[['Date', 'label', 'sentiment_anomaly']].tail(20))

In [ ]:
def lstm_autoencoder_detect(signal_df, anomalies_df, window=10, hidden_units=80, epochs=35, batch_size=64):
    """
    signal_df: ['timestamp','value'] DataFrame (value = Volume)
    anomalies_df: ['start','end'] DataFrame (실제 이상치, Unix timestamp)
    window: 슬라이딩 윈도우 길이 (예: 10, 50, 100, 250 등 마음대로 실험)
    hidden_units: LSTM 은닉 유닛 수 (논문에서는 80)
    epochs: 학습 에포크 수 (예: 10, 20, 35, 50, 70 등)
    batch_size: 배치 크기 (예: 32, 64 등)

    returns:
      - detected_df: 탐지된 이상치 구간 DataFrame(['start','end'])
      - metrics: {'precision', 'recall', 'f1'} (포인트 단위 평가)
      - elapsed: 실행 시간(초, float)
    """
    # 1) 시계열(array) 준비
    values = signal_df['value'].values.reshape(-1,1).astype('float32')  # shape=(n,1)
    timestamps = signal_df['timestamp'].values  # shape=(n,)
    n = len(values)

    # 2) MinMaxScaler로 0~1 정규화
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)  # shape=(n,1)

    # 3) 슬라이딩 윈도우 시퀀스 생성 (shape = (n-window+1, window, 1))
    X = []
    for i in range(n - window + 1):
        X.append(scaled[i : i+window])
    X = np.array(X)

    # 4) LSTM Autoencoder 모델 정의
    model = Sequential([
        LSTM(hidden_units, activation='relu', input_shape=(window,1)),
        RepeatVector(window),
        LSTM(hidden_units, activation='relu', return_sequences=True),
        TimeDistributed(Dense(1))
    ])
    model.compile(optimizer='adam', loss='mse')

    # 5) 모델 학습
    start_time = time.time()
    model.fit(X, X, epochs=epochs, batch_size=batch_size, shuffle=False, verbose=0)

    # 6) 재구성값 예측 및 MSE 계산 (각 시퀀스별)
    X_pred = model.predict(X, batch_size=batch_size, verbose=0)
    mse = np.mean(np.square(X_pred - X), axis=(1,2))  # shape=(n-window+1,)

    # 7) 동적 Threshold 설정: 전체 MSE 분포의 mean + 3*std
    thr = np.mean(mse) + 3*np.std(mse)

    # 8) MSE > thr 인 시퀀스의 “마지막 인덱스(i+window-1)” 플래그(1)
    detected_flags = np.zeros(n, dtype=int)
    for i, err in enumerate(mse):
        if err > thr:
            idx = i + window - 1
            if idx < n:
                detected_flags[idx] = 1
    elapsed = time.time() - start_time

    # 9) 실제 이상치 구간에 해당하는 포인트 라벨 생성
    is_true_label = np.zeros(n, dtype=int)
    for _, row in anomalies_df.iterrows():
        mask = (timestamps >= row['start']) & (timestamps <= row['end'])
        is_true_label[mask] = 1

    # 10) detected_flags(0/1) → 연속 구간(['start','end'])으로 묶기
    detected_windows = []
    in_anom = False
    for i in range(n):
        if detected_flags[i] == 1 and not in_anom:
            s_idx = i
            in_anom = True
        if (detected_flags[i] == 0 or i == n-1) and in_anom:
            e_idx = i-1 if detected_flags[i] == 0 else i
            detected_windows.append((timestamps[s_idx], timestamps[e_idx]))
            in_anom = False
    detected_df = pd.DataFrame(detected_windows, columns=['start','end'])

    # 11) 포인트 단위 Precision / Recall / F1 계산
    precision = precision_score(is_true_label, detected_flags, zero_division=0)
    recall    = recall_score(is_true_label, detected_flags, zero_division=0)
    f1        = f1_score(is_true_label, detected_flags, zero_division=0)
    metrics = {"precision": precision, "recall": recall, "f1": f1}

    return detected_df, metrics, elapsed

In [ ]:
# 실험할 윈도우 크기 목록
window_list = [5, 10, 20, 50, 100, 250]
# 실험할 에포크 수 목록
epoch_list = [10, 35, 70]

results_auto = []

for ticker in tickers:
    sig_df   = all_signals[ticker]
    anom_df  = all_anomalies[ticker]

    for w in window_list:
        # “시퀀스 X 차원 = (n-w+1, w, 1)” 이므로, w가 너무 크면 n-w+1이 작아져 학습이 불안정해질 수 있음
        if w >= len(sig_df):
            continue

        for e in epoch_list:
            detected_df, metrics, elapsed = lstm_autoencoder_detect(
                sig_df,
                anom_df,
                window=w,
                hidden_units=80,
                epochs=e,
                batch_size=64
            )
            results_auto.append({
                "Stock": ticker,
                "Window": w,
                "Epochs": e,
                "Precision": round(metrics['precision'], 3),
                "Recall":    round(metrics['recall'],    3),
                "F1":        round(metrics['f1'],        3),
                "Elapsed":   round(elapsed,              2)
            })

df_auto = pd.DataFrame(results_auto)
print("\n=== LSTM Autoencoder 전체 실험 결과 ===")
print(df_auto.to_string(index=False))

In [ ]:
# 1) df_auto 에서 종목별로 F1-score가 가장 높은 조합(행)을 추출
best_auto_rows = df_auto.loc[df_auto.groupby("Stock")["F1"].idxmax()].reset_index(drop=True)

# 2) 컬럼 순서 및 이름 변경
best_auto_table = best_auto_rows[["Stock", "Window", "Epochs", "F1", "Elapsed"]].rename(columns={
    "Window": "Best window",
    "Epochs": "Best epochs",
    "F1": "F1-score",
    "Elapsed": "Elapsed time (s)"
})

print("\n=== Table 2 (LSTM Autoencoder 기준) – Best approach for each data set ===")
print(best_auto_table.to_string(index=False))

In [ ]:
# best_windows, best_epochs 사전 초기화
best_windows = {}
best_epochs = {}

# best_auto_table에서 반복문으로 추출하여 저장
for idx, row in best_auto_table.iterrows():
    stock = row['Stock']
    best_windows[stock] = int(row['Best window'])
    best_epochs[stock] = int(row['Best epochs'])

# 확인용 출력
print("best_windows =", best_windows)
print("best_epochs =", best_epochs)

In [ ]:
def plot_autoencoder_results(ticker, df_original, anomalies_df, best_window, best_epochs, detected_windows_df):
    """
    ticker: 종목명 (문자열)
    df_original: 원본 DataFrame (['Date','Volume','timestamp', …])
    anomalies_df: 실제 이상치 구간 DataFrame(['start','end'])
    best_window, best_epochs: Best으로 선택된 윈도우 크기와 에포크 수
    detected_windows_df: 탐지된 이상치 구간 DataFrame(['start','end'])
    """
    # 1) 날짜 인덱스로 설정
    df_plot = df_original.copy()
    df_plot['date'] = pd.to_datetime(df_plot['Date'])
    df_plot.set_index('date', inplace=True)

    # 2) 거래량 선 그래프
    plt.figure(figsize=(12,4))
    plt.plot(df_plot.index, df_plot['Volume'], label="Volume", color='lightgray')
    plt.title(f"Traded volumes for {ticker} → LSTM AE (w={best_window}, e={best_epochs})", fontsize=12)
    plt.ylabel("Volume")
    plt.xlabel("Time")

    # 3) 실제 이상치(녹색)
    for idx, row in anomalies_df.iterrows():
        s_dt = datetime.datetime.fromtimestamp(row['start'])
        e_dt = datetime.datetime.fromtimestamp(row['end'])
        plt.axvspan(s_dt, e_dt, color='green', alpha=0.3, label="Real anomalies" if idx==0 else "")

    # 4) 탐지된 이상치(빨간색)
    for idx, row in detected_windows_df.iterrows():
        s_dt = datetime.datetime.fromtimestamp(row['start'])
        e_dt = datetime.datetime.fromtimestamp(row['end'])
        plt.axvspan(s_dt, e_dt, color='red', alpha=0.2, label="Detected anomalies" if idx==0 else "")

    # 5) 범례 및 x축 포맷
    plt.legend(loc='upper left', fontsize=9)
    plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
for idx, row in best_auto_table.iterrows():
    ticker      = row['Stock']
    best_window = int(row['Best window'])
    best_epochs = int(row['Best epochs'])

    df_orig   = all_data[ticker]
    anom_df   = all_anomalies[ticker]

    # 1) Best (window, epochs)으로 탐지 수행
    detected_df, _, _ = lstm_autoencoder_detect(
        all_signals[ticker],
        anom_df,
        window=best_window,
        hidden_units=80,
        epochs=best_epochs,
        batch_size=64
    )

    # 2) 시각화
    plot_autoencoder_results(ticker, df_orig, anom_df, best_window, best_epochs, detected_df)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve

In [ ]:
def make_lstm_score_df(signal_df, mse_score, window):
    """
    LSTM Autoencoder의 시퀀스별 mse_score(재구성 오차)를 원래 시계열에 매핑
    - signal_df: ['timestamp', 'value']
    - mse_score: shape = (n-window+1,)
    - window: 윈도우 크기
    """
    # mse_score는 윈도우 마지막 인덱스에 해당함
    date_list = [datetime.datetime.fromtimestamp(ts).date() for ts in signal_df['timestamp'].values]
    df = pd.DataFrame({'date': date_list, 'lstm_score': np.nan})
    for i, score in enumerate(mse_score):
        idx = i + window - 1
        if idx < len(df):
            df.at[idx, 'lstm_score'] = score
    df['lstm_score'] = df['lstm_score'].fillna(0)
    return df

In [ ]:
def make_sentiment_df(df_sent):
    # df_sent: ['Date', 'sentiment_mean', ...]
    # 추가로 긍정비율/텍스트량 등 피처도 만들기 (원하면 summary 등 추가!)
    df = df_sent.copy()
    df['positive_ratio'] = (df['sentiment_mean'] > 0).astype(int)
    df['text_length'] = df.get('text', '').apply(lambda x: len(x) if isinstance(x, str) else 0) if 'text' in df else 0
    df = df[['Date', 'sentiment_mean', 'positive_ratio', 'text_length']]
    df.columns = ['date', 'sentiment_mean', 'positive_ratio', 'text_length']
    df['date'] = pd.to_datetime(df['date']).dt.date
    return df

In [ ]:
def make_label_df(signal_df, anomalies_df):
    # 1일 단위 날짜별로 실제 이상치(1)/정상(0) 라벨 생성
    dates = [datetime.datetime.fromtimestamp(ts).date() for ts in signal_df['timestamp'].values]
    labels = []
    for ts in signal_df['timestamp'].values:
        label = 0
        for _, row in anomalies_df.iterrows():
            if ts >= row['start'] and ts <= row['end']:
                label = 1
                break
        labels.append(label)
    df = pd.DataFrame({'date': dates, 'label': labels})
    return df

In [ ]:
# (1) LSTM 점수 추출
from sklearn.preprocessing import MinMaxScaler
import time
from keras.models import Sequential
from keras.layers import LSTM, Dense, RepeatVector, TimeDistributed

def lstm_autoencoder_detect(signal_df, anomalies_df, window=10, hidden_units=80, epochs=35, batch_size=64, return_mse_score=False):
    values = signal_df['value'].values.reshape(-1,1).astype('float32')
    timestamps = signal_df['timestamp'].values
    n = len(values)
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)
    X = np.array([scaled[i:i+window] for i in range(n-window+1)])
    model = Sequential([
        LSTM(hidden_units, activation='relu', input_shape=(window,1)),
        RepeatVector(window),
        LSTM(hidden_units, activation='relu', return_sequences=True),
        TimeDistributed(Dense(1))
    ])
    model.compile(optimizer='adam', loss='mse')
    start_time = time.time()
    model.fit(X, X, epochs=epochs, batch_size=batch_size, shuffle=False, verbose=0)
    X_pred = model.predict(X, batch_size=batch_size, verbose=0)
    mse = np.mean(np.square(X_pred - X), axis=(1,2))
    thr = np.mean(mse) + 3*np.std(mse)
    detected_flags = np.zeros(n, dtype=int)
    for i, err in enumerate(mse):
        idx = i + window - 1
        if err > thr and idx < n:
            detected_flags[idx] = 1
    elapsed = time.time() - start_time
    is_true_label = np.zeros(n, dtype=int)
    for _, row in anomalies_df.iterrows():
        mask = (timestamps >= row['start']) & (timestamps <= row['end'])
        is_true_label[mask] = 1
    detected_windows = []
    in_anom = False
    for i in range(n):
        if detected_flags[i] == 1 and not in_anom:
            s_idx = i
            in_anom = True
        if (detected_flags[i] == 0 or i == n-1) and in_anom:
            e_idx = i-1 if detected_flags[i] == 0 else i
            detected_windows.append((timestamps[s_idx], timestamps[e_idx]))
            in_anom = False
    detected_df = pd.DataFrame(detected_windows, columns=['start','end'])
    precision = precision_score(is_true_label, detected_flags, zero_division=0)
    recall    = recall_score(is_true_label, detected_flags, zero_division=0)
    f1        = f1_score(is_true_label, detected_flags, zero_division=0)
    metrics = {"precision": precision, "recall": recall, "f1": f1}
    if return_mse_score:
        return detected_df, metrics, elapsed, mse
    else:
        return detected_df, metrics, elapsed


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve
from sklearn.model_selection import train_test_split

# (예시) lstm_score_df, sentiment_df, label_df가 stock별로 이미 준비되어 있다고 가정
# lstm_score_df: ['date', 'lstm_score']
# sentiment_df:  ['date', 'sentiment_mean', 'positive_ratio', 'text_length']
# label_df:      ['date', 'label']

# 날짜 범위 설정
date_min = min(
    lstm_score_df['date'].min(),
    sentiment_df['date'].min(),
    label_df['date'].min()
)
date_max = max(
    lstm_score_df['date'].max(),
    sentiment_df['date'].max(),
    label_df['date'].max()
)
date_list = pd.date_range(date_min, date_max, freq='D')

# (1) 날짜 기준 df_merged 생성 (누락일은 0으로 채움)
df_merged = pd.DataFrame({'date': date_list.date})
df_merged = df_merged.merge(lstm_score_df, on='date', how='left')
df_merged = df_merged.merge(sentiment_df, on='date', how='left')
df_merged = df_merged.merge(label_df, on='date', how='left')
df_merged = df_merged.fillna(0)
df_merged['date'] = pd.to_datetime(df_merged['date'])

# (2) feature/target 분리
feature_cols = ['lstm_score', 'sentiment_mean', 'positive_ratio', 'text_length']
X = df_merged[feature_cols].values
y = df_merged['label'].values

# (3) 클래스 불균형에 따른 pos_weight
n_pos = np.sum(y == 1)
n_neg = np.sum(y == 0)
pos_weight = n_neg / (n_pos + 1e-8) if n_pos > 0 else 1

# (4) 학습/검증 분리
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# (5) XGBoost 모델 학습 (구버전: early_stopping_rounds 없이)
model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    scale_pos_weight=pos_weight,
    random_state=42
)
model.fit(X_train, y_train)

# (6) 임계값 튜닝 (F1-score 최대화)
val_probs = model.predict_proba(X_val)[:,1]
prec, rec, thr = precision_recall_curve(y_val, val_probs)
f1s = 2 * prec * rec / (prec + rec + 1e-8)
best_thr = thr[np.argmax(f1s)]
print(f"Best threshold (F1-max): {best_thr:.3f}")

y_pred = (val_probs >= best_thr).astype(int)

# (7) 평가 지표
print("XGBoost 통합분류 결과")
print(f"Precision: {precision_score(y_val, y_pred):.3f}")
print(f"Recall:    {recall_score(y_val, y_pred):.3f}")
print(f"F1-score:  {f1_score(y_val, y_pred):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, val_probs):.3f}")

# (8) PR Curve (optional)
import matplotlib.pyplot as plt
plt.figure(figsize=(5,4))
plt.plot(rec, prec, label="PR curve")
plt.scatter(rec[np.argmax(f1s)], prec[np.argmax(f1s)], color='red', label="Best F1")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (Validation)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve, roc_auc_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 종목 리스트
stocks = ['AEMD', 'TRBO', 'NBDR', 'GME', 'APPB']

# 결과 저장용
result_list = []

# PR Curve 그리기용
plt.figure(figsize=(10, 6))

for stock in stocks:
    # ① 개별 종목 데이터 준비
    lstm_score_df = all_lstm_score[stock]
    sentiment_df  = all_sentiment[stock]
    label_df      = all_label[stock]

    # 날짜 통합 범위
    date_min = min(
        lstm_score_df['date'].min(),
        sentiment_df['date'].min(),
        label_df['date'].min()
    )
    date_max = max(
        lstm_score_df['date'].max(),
        sentiment_df['date'].max(),
        label_df['date'].max()
    )
    date_list = pd.date_range(date_min, date_max, freq='D')

    df_merged = pd.DataFrame({'date': date_list.date})
    df_merged = df_merged.merge(lstm_score_df, on='date', how='left')
    df_merged = df_merged.merge(sentiment_df, on='date', how='left')
    df_merged = df_merged.merge(label_df, on='date', how='left')
    df_merged = df_merged.fillna(0)
    df_merged['date'] = pd.to_datetime(df_merged['date'])

    # feature/target 분리
    feature_cols = ['lstm_score', 'sentiment_mean', 'positive_ratio', 'text_length']
    X = df_merged[feature_cols].values
    y = df_merged['label'].values

    n_pos = np.sum(y == 1)
    n_neg = np.sum(y == 0)
    pos_weight = n_neg / (n_pos + 1e-8) if n_pos > 0 else 1

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        use_label_encoder=False,
        scale_pos_weight=pos_weight,
        random_state=42
    )
    model.fit(X_train, y_train)

    # F1-score 기준 임계값 최적화
    val_probs = model.predict_proba(X_val)[:, 1]
    prec, rec, thr = precision_recall_curve(y_val, val_probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-8)
    best_thr = thr[np.argmax(f1s)]
    y_pred = (val_probs >= best_thr).astype(int)

    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, val_probs)

    result_list.append([
        stock, round(precision, 3), round(recall, 3), round(f1, 3), round(auc, 3), round(best_thr, 3)
    ])

    # PR curve plot
    plt.plot(rec, prec, label=f'{stock} (F1={f1:.2f})')
    plt.scatter(rec[np.argmax(f1s)], prec[np.argmax(f1s)], marker='o', color='black')
    plt.text(rec[np.argmax(f1s)], prec[np.argmax(f1s)], stock, fontsize=9)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (Validation)")
plt.legend()
plt.tight_layout()
plt.show()

# 종목별 평가표 출력
result_df = pd.DataFrame(result_list, columns=['Stock', 'Precision', 'Recall', 'F1-score', 'ROC-AUC', 'Best Thr'])
print("\n=== XGBoost 통합 이상 탐지 평가표 ===")
print(result_df.to_string(index=False))


In [ ]:
import datetime
import pandas as pd
import numpy as np

# (1) LSTM 점수 DF
def make_lstm_score_df(signal_df, mse_score, window):
    # signal_df: ['timestamp', 'value']
    # mse_score: (n-window+1, )
    date_list = [datetime.datetime.fromtimestamp(ts).date() for ts in signal_df['timestamp'].values]
    df = pd.DataFrame({'date': date_list, 'lstm_score': np.nan})
    for i, score in enumerate(mse_score):
        idx = i + window - 1
        if idx < len(df):
            df.at[idx, 'lstm_score'] = score
    df['lstm_score'] = df['lstm_score'].fillna(0)
    return df

# (2) 감성 DF (추가 피처 포함)
def make_sentiment_df(df_sent):
    df = df_sent.copy()
    df['positive_ratio'] = (df['sentiment_mean'] > 0).astype(int)
    df['text_length'] = df.get('text', '').apply(lambda x: len(x) if isinstance(x, str) else 0) if 'text' in df else 0
    df = df[['Date', 'sentiment_mean', 'positive_ratio', 'text_length']]
    df.columns = ['date', 'sentiment_mean', 'positive_ratio', 'text_length']
    df['date'] = pd.to_datetime(df['date']).dt.date
    return df

# (3) 라벨 DF (실제 이상치 구간: 1, 나머지 0)
def make_label_df(signal_df, anomalies_df):
    dates = [datetime.datetime.fromtimestamp(ts).date() for ts in signal_df['timestamp'].values]
    labels = []
    for ts in signal_df['timestamp'].values:
        label = 0
        for _, row in anomalies_df.iterrows():
            if ts >= row['start'] and ts <= row['end']:
                label = 1
                break
        labels.append(label)
    df = pd.DataFrame({'date': dates, 'label': labels})
    return df


In [ ]:
# 종목 리스트
stocks = ['AEMD', 'TRBO', 'NBDR', 'GME', 'APPB']

# best_window, best_epochs는 이미 선정되어 있다고 가정 (종목별 dict)
# 예시: best_windows = {'AEMD': 20, ...}
# 각 종목별 LSTM 학습 + mse_score 뽑기 (예시)
from keras.models import Sequential
from keras.layers import LSTM, Dense, RepeatVector, TimeDistributed
from sklearn.preprocessing import MinMaxScaler
import time

def get_mse_score(signal_df, anomalies_df, window=10, hidden_units=80, epochs=35, batch_size=64):
    values = signal_df['value'].values.reshape(-1,1).astype('float32')
    timestamps = signal_df['timestamp'].values
    n = len(values)
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)
    X = np.array([scaled[i:i+window] for i in range(n-window+1)])
    model = Sequential([
        LSTM(hidden_units, activation='relu', input_shape=(window,1)),
        RepeatVector(window),
        LSTM(hidden_units, activation='relu', return_sequences=True),
        TimeDistributed(Dense(1))
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(X, X, epochs=epochs, batch_size=batch_size, shuffle=False, verbose=0)
    X_pred = model.predict(X, batch_size=batch_size, verbose=0)
    mse = np.mean(np.square(X_pred - X), axis=(1,2))
    return mse

# 딕셔너리로 저장
all_lstm_score = {}
all_sentiment = {}
all_label = {}

for stock in stocks:
    print(f"Preparing data for {stock}...")

    # ① LSTM 재구성오차
    window = best_windows[stock]
    epochs = best_epochs[stock]
    mse_score = get_mse_score(all_signals[stock], all_anomalies[stock], window=window, epochs=epochs)
    lstm_score_df = make_lstm_score_df(all_signals[stock], mse_score, window)
    all_lstm_score[stock] = lstm_score_df

    # ② 감성 피처 (이미 run_finbert/daily_sentiment/detect_sentiment_anomaly까지 처리한 결과를 사용)
    df_sent = df_sent_dict[stock]   # ['Date', 'sentiment_mean', ...]
    sentiment_df = make_sentiment_df(df_sent)
    all_sentiment[stock] = sentiment_df

    # ③ 라벨
    label_df = make_label_df(all_signals[stock], all_anomalies[stock])
    all_label[stock] = label_df


In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve
import pandas as pd
import numpy as np

results = []
for stock in stocks:
    # 1) 날짜 기준 merge
    date_min = min(
        all_lstm_score[stock]['date'].min(),
        all_sentiment[stock]['date'].min(),
        all_label[stock]['date'].min()
    )
    date_max = max(
        all_lstm_score[stock]['date'].max(),
        all_sentiment[stock]['date'].max(),
        all_label[stock]['date'].max()
    )
    date_list = pd.date_range(date_min, date_max, freq='D')
    df_merged = pd.DataFrame({'date': date_list.date})
    df_merged = df_merged.merge(all_lstm_score[stock], on='date', how='left')
    df_merged = df_merged.merge(all_sentiment[stock], on='date', how='left')
    df_merged = df_merged.merge(all_label[stock], on='date', how='left')
    df_merged = df_merged.fillna(0)
    df_merged['date'] = pd.to_datetime(df_merged['date'])

    # 2) feature/target
    feature_cols = ['lstm_score', 'sentiment_mean', 'positive_ratio', 'text_length']
    X = df_merged[feature_cols].values
    y = df_merged['label'].values

    # 3) 클래스 불균형 보정
    n_pos = np.sum(y == 1)
    n_neg = np.sum(y == 0)
    pos_weight = n_neg / (n_pos + 1e-8) if n_pos > 0 else 1

    # 4) 학습/검증 분리
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    # 5) XGBoost 학습
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        use_label_encoder=False,
        scale_pos_weight=pos_weight,
        random_state=42
    )
    model.fit(X_train, y_train)

    # 6) 임계값 튜닝 (F1-max)
    val_probs = model.predict_proba(X_val)[:,1]
    prec, rec, thr = precision_recall_curve(y_val, val_probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-8)
    best_thr = thr[np.argmax(f1s)]
    y_pred = (val_probs >= 0.5).astype(int)

    # 7) 평가 기록
    results.append({
        "Stock": stock,
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1-score": f1_score(y_val, y_pred),
        "Best threshold": best_thr
    })

# 결과표 출력
results_df = pd.DataFrame(results)
print(results_df)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve
import xgboost as xgb


In [ ]:
df_merged_dict = {}
for stock in stocks:
    lstm_score_df = all_lstm_score[stock]
    sentiment_df  = all_sentiment[stock]
    label_df      = all_label[stock]
    date_min = min(lstm_score_df['date'].min(), sentiment_df['date'].min(), label_df['date'].min())
    date_max = max(lstm_score_df['date'].max(), sentiment_df['date'].max(), label_df['date'].max())
    date_list = pd.date_range(date_min, date_max, freq='D')
    merged = pd.DataFrame({'date': date_list.date})
    merged = merged.merge(lstm_score_df, on='date', how='left')
    merged = merged.merge(sentiment_df, on='date', how='left')
    merged = merged.merge(label_df, on='date', how='left')
    merged = merged.fillna(0)
    merged['date'] = pd.to_datetime(merged['date'])
    df_merged_dict[stock] = merged

In [ ]:
feature_cols = ['lstm_score', 'sentiment_mean', 'positive_ratio', 'text_length']
best_thresholds = {}
xgb_model_dict = {}

for stock in stocks:
    df = df_merged_dict[stock]
    X = df[feature_cols].values
    y = df['label'].values

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    n_pos = np.sum(y_train == 1)
    n_neg = np.sum(y_train == 0)
    pos_weight = n_neg / (n_pos + 1e-8) if n_pos > 0 else 1

    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        use_label_encoder=False,
        scale_pos_weight=pos_weight,
        random_state=42
    )
    model.fit(X_train, y_train)
    xgb_model_dict[stock] = model

    val_probs = model.predict_proba(X_val)[:,1]
    prec, rec, thr = precision_recall_curve(y_val, val_probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-8)
    best_thr = thr[np.argmax(f1s)] if len(thr) > 0 else 0.5
    best_thresholds[stock] = best_thr


In [ ]:
def extract_detected_anomalies_from_xgb(df, date_col='date', label_col='xgb_pred'):
    detected = []
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)
    in_anom = False
    for i, row in df.iterrows():
        if row[label_col] == 1 and not in_anom:
            s = row[date_col]
            in_anom = True
        if (row[label_col] == 0 or i == len(df) - 1) and in_anom:
            e = df[date_col][i-1 if row[label_col]==0 else i]
            detected.append((s, e))
            in_anom = False
    return detected

In [ ]:
detected_anomalies_xgb = {}
for stock in stocks:
    df_pred = df_xgb_pred[stock]
    detected_anomalies_xgb[stock] = extract_detected_anomalies_from_xgb(df_pred, date_col='date', label_col='xgb_pred')


In [ ]:
for stock in stocks:
    # df_merged_dict[stock]와 xgb_model_dict[stock] 등이 제대로 생성되어야 함
    this_df = df_merged_dict[stock].copy()
    model = xgb_model_dict[stock]
    best_thr = best_thresholds[stock]
    # XGBoost 예측
    this_df['xgb_prob'] = model.predict_proba(this_df[feature_cols].values)[:,1]
    this_df['xgb_pred'] = (this_df['xgb_prob'] >= best_thr).astype(int)
    df_xgb_pred[stock] = this_df[['date', 'xgb_pred', 'xgb_prob']]


In [ ]:
df_xgb_pred.keys()

In [ ]:
#----------------------------
def plot_xgb_detection_vs_real(all_stock_data, all_anomalies, detected_anomalies_xgb):
    stocks = list(all_stock_data.keys())
    n_stocks = len(stocks)
    fig, axs = plt.subplots(n_stocks, 1, figsize=(10, 12), sharex=True)
    for idx, stock in enumerate(stocks):
        ax = axs[idx]
        df = all_stock_data[stock]
        ax.plot(df['Date'], df['Volume'], label='Volume', color='royalblue', lw=1)

        # 실제 이상치(녹색)
        for _, row in all_anomalies[stock].iterrows():
            ax.axvspan(pd.to_datetime(row['start'], unit='s'), pd.to_datetime(row['end'], unit='s'), color='green', alpha=0.2)

        # XGBoost 탐지(빨간색)
        for start, end in detected_anomalies_xgb.get(stock, []):
            ax.axvspan(start, end, color='red', alpha=0.18)

        company = {
            "TRBO": "Turbo Global Partners, Inc.",
            "APPB": "Applied Biosciences Corp",
            "AEMD": "Aethlon Medical, Inc.",
            "NBDR": "No Borders, Inc.",
            "GME": "GameStop"
        }.get(stock, stock)
        ax.set_title(f'Traded volumes for {company} (“{stock}”)', fontsize=11)
        ax.set_ylabel('Volume')

        if idx == 0:
            green_patch = mpatches.Patch(color='green', alpha=0.2, label='Real anomalies')
            red_patch = mpatches.Patch(color='red', alpha=0.18, label='XGBoost-detected')
            ax.legend(handles=[red_patch, green_patch], loc='upper left', fontsize=8)

        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.grid(True)
    axs[-1].set_xlabel('Date')
    plt.tight_layout(rect=[0, 0, 1, 1])
    plt.show()


In [ ]:
plot_xgb_detection_vs_real(all_stock_data, all_anomalies, detected_anomalies_xgb)